# 🌿 CropLogic — AI-Driven Smart Agriculture DSS
### Complete Pipeline Walkthrough
**Author:** Abinaya V (CB.PS.I5DAS22102)  
**Dept:** Mathematics, Amrita Vishwa Vidyapeetham  

This notebook runs the entire CropLogic pipeline:
1. Install dependencies
2. Fetch real NDVI data (NASA POWER API)
3. Fetch weather (Open-Meteo API)
4. Z-score stress detection
5. Disease risk model (GBM)
6. CNN disease classifier (ResNet-50 on PlantVillage)
7. Recommendation engine
8. Launch Streamlit dashboard

## Step 1 — Install Dependencies

In [ ]:
# Detect environment
import sys, os
IN_COLAB  = 'google.colab' in sys.modules
IN_KAGGLE = os.environ.get('KAGGLE_KERNEL_RUN_TYPE') is not None
print(f'Colab: {IN_COLAB} | Kaggle: {IN_KAGGLE}')

!pip install -q streamlit tensorflow scikit-learn plotly folium streamlit-folium \
    requests Pillow pandas numpy scipy joblib tqdm opendatasets

## Step 2 — Clone / Setup Project

In [ ]:
# If running from a zip upload, extract it first:
# import zipfile
# with zipfile.ZipFile('croplogic.zip') as z: z.extractall('.')
# os.chdir('croplogic')

# Otherwise, if cloning from GitHub:
# !git clone https://github.com/YOUR_REPO/croplogic.git
# os.chdir('croplogic')

# Verify structure
for root, dirs, files in os.walk('.'):
    dirs[:] = [d for d in dirs if d not in ['__pycache__','.git','models']]
    level = root.replace('.', '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    for f in files:
        print(f'{indent}  {f}')

## Step 3 — Fetch Real-Time NDVI (NASA POWER API)

In [ ]:
import sys
sys.path.insert(0, '.')

from modules.data_fetcher import build_ndvi_series, fetch_weather

# Define a demo field
field = {
    'id': 1,
    'name': 'Kaveri Rice Block',
    'crop': 'Rice (Paddy)',
    'lat': 10.79,
    'lon': 77.00,
    'area': 3.2,
    'sow': '2024-07-01',
}

print('Fetching NDVI data from NASA POWER API...')
ndvi_df = build_ndvi_series(field, days=30)
print(f'\n✓ NDVI series: {len(ndvi_df)} days')
ndvi_df.tail()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np

fig, axes = plt.subplots(2, 1, figsize=(14, 8), facecolor='#0d1117')
for ax in axes:
    ax.set_facecolor('#161b22')
    ax.tick_params(colors='#8b949e')
    ax.spines['bottom'].set_color('#2d333b')
    ax.spines['left'].set_color('#2d333b')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

mu    = ndvi_df['mu'].iloc[0]
sigma = ndvi_df['sigma'].iloc[0]
tau   = ndvi_df['tau'].iloc[0]

# NDVI plot
ax1 = axes[0]
ax1.plot(ndvi_df['date'], ndvi_df['ndvi'], color='#3fb950', linewidth=2, label='NDVI')
ax1.fill_between(ndvi_df['date'], 0, ndvi_df['ndvi'], alpha=0.12, color='#3fb950')
ax1.axhline(mu + tau*sigma, color='#f85149', linestyle='--', linewidth=1, label=f'μ+τσ ({mu+tau*sigma:.3f})')
ax1.axhline(mu,             color='white',   linestyle=':', linewidth=0.8, alpha=0.4, label=f'μ ({mu})')
ax1.axhline(mu - tau*sigma, color='#f85149', linestyle='--', linewidth=1, label=f'μ-τσ ({mu-tau*sigma:.3f})')
ax1.set_ylabel('NDVI', color='#8b949e')
ax1.set_title(f"NDVI Time Series — {field['name']}", color='#e6edf3', pad=12)
ax1.legend(facecolor='#1c2330', edgecolor='#2d333b', labelcolor='#8b949e')
ax1.set_ylim(0, 1)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))

# Z-score plot
ax2 = axes[1]
colors = ['#3fb950' if abs(z)<=1 else '#d29922' if abs(z)<=tau else '#f85149'
          for z in ndvi_df['zscore']]
ax2.bar(ndvi_df['date'], ndvi_df['zscore'], color=colors, width=0.8)
ax2.axhline( tau, color='#f85149', linestyle='--', linewidth=1, label=f'+τ={tau}')
ax2.axhline(-tau, color='#f85149', linestyle='--', linewidth=1, label=f'-τ={tau}')
ax2.axhline(0, color='white', linewidth=0.5, alpha=0.3)
ax2.set_ylabel('Z-Score', color='#8b949e')
ax2.set_title('Z-Score Stress Detection', color='#e6edf3', pad=12)
ax2.legend(facecolor='#1c2330', edgecolor='#2d333b', labelcolor='#8b949e')
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))

plt.tight_layout()
plt.savefig('outputs/ndvi_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Plot saved to outputs/ndvi_analysis.png')

## Step 4 — Weather Data (Open-Meteo)

In [ ]:
print('Fetching weather from Open-Meteo...')
wx = fetch_weather(field['lat'], field['lon'])

if wx:
    cur = wx['current']
    print(f"\n Current Conditions:")
    print(f"  Temperature : {cur['temperature_2m']}°C")
    print(f"  Humidity    : {cur['relative_humidity_2m']}%")
    print(f"  Precipitation: {cur['precipitation']} mm")
    print(f"  Wind        : {cur['wind_speed_10m']} km/h")
    print(f"  Cloud Cover : {cur['cloud_cover']}%")

    import pandas as pd
    df_wx = pd.DataFrame({'Date': wx['daily']['time'],
                          'T max (°C)': wx['daily']['temp_max'],
                          'T min (°C)': wx['daily']['temp_min'],
                          'Rain (mm)': wx['daily']['precip_sum']})
    print('\n7-Day Forecast:')
    print(df_wx.to_string(index=False))
else:
    print('Weather unavailable — check connection')

## Step 5 — Full Field Analysis (Z-score + Risk)

In [ ]:
from modules.ndvi_analysis import analyse_field
import json

analysis = analyse_field(field, ndvi_df, wx)

print('=== Field Analysis Report ===')
print(f"Field     : {analysis['field']['name']}")
print(f"NDVI Now  : {analysis['ndvi_now']}")
print(f"Z-Score   : {analysis['zscore']['zscore']:+.3f}")
print(f"Stress    : {analysis['zscore']['stress_level']}")
print(f"Alert     : {analysis['zscore']['alert']}")
print(f"Trend β₁  : {analysis['trend']['beta1']:+.6f}/day (p={analysis['trend']['pvalue']:.4f})")
print(f"Trend Alert: {analysis['trend']['trend_alert']}")
print(f"Disease Risk: {analysis['risk']['level']} (P={analysis['risk']['probability']:.3f})")
print(f"\nRisk Factors:")
for factor, score in analysis['risk']['factors'].items():
    bar = '█' * int(score * 40)
    print(f"  {factor:<14} {score:.3f}  {bar}")

## Step 6 — Train GBM Risk Model

In [ ]:
from modules.risk_model import train_risk_model, RiskModel
import os

os.makedirs('models', exist_ok=True)

print('Training Gradient Boosting risk classifier...')
gbm_model, scaler = train_risk_model(save_dir='models')

# Test inference
rm = RiskModel('models/risk_gbm.joblib', 'models/scaler.joblib')
result = rm.predict(
    ndvi=analysis['ndvi_now'],
    beta1=analysis['trend']['beta1'],
    zscore=analysis['zscore']['zscore'],
    temp=analysis['weather']['temperature_c'],
    humidity=analysis['weather']['humidity_pct'],
    rain_7d=analysis['weather']['rain_7d_mm'],
    days_since_sow=90,
    crop=field['crop']
)
print(f"\nGBM Risk Prediction: {result['level']} (P={result['probability']:.3f})")
print(f"Probabilities: {result['proba_all']}")

## Step 7 — Download PlantVillage & Train CNN (GPU recommended)

In [ ]:
# Download dataset
# Option A — opendatasets (prompts for Kaggle credentials)
# import opendatasets as od
# od.download('https://www.kaggle.com/datasets/vipoooool/new-plant-diseases-dataset')
# data_dir = 'new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)'

# Option B — Kaggle CLI (set up kaggle.json first)
# !kaggle datasets download -d vipoooool/new-plant-diseases-dataset -p data/
# !unzip -q data/new-plant-diseases-dataset.zip -d data/plantvillage
# data_dir = 'data/plantvillage/New Plant Diseases Dataset(Augmented)'

data_dir = 'data/plantvillage/New Plant Diseases Dataset(Augmented)'

import os
if os.path.isdir(data_dir):
    print(f'Dataset found at {data_dir}')
    from modules.cnn_model import train
    os.makedirs('models', exist_ok=True)
    model, h1, h2 = train(data_dir, save_dir='models')
    print(f'\nFinal val accuracy Phase 2: {max(h2.history["val_accuracy"]):.4f}')
else:
    print(f'Dataset not found at: {data_dir}')
    print('Please download PlantVillage first (see instructions above)')

## Step 8 — CNN Inference on Sample Image

In [ ]:
from modules.cnn_model import DiseaseClassifier
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

# Load classifier (will use heuristic demo mode if model not trained yet)
clf = DiseaseClassifier(
    model_path       = 'models/resnet50_plantvillage.h5'    if os.path.exists('models/resnet50_plantvillage.h5') else None,
    class_index_path = 'models/class_indices.json'          if os.path.exists('models/class_indices.json')      else None,
)

# To test with your own image:
#   img_path = 'path/to/leaf.jpg'
#   results  = clf.predict(img_path, top_k=5)

# Demo with synthetic image (random noise — real model would produce meaningful results)
fake_img = np.random.randint(50, 200, (224, 224, 3), dtype=np.uint8)
results  = clf.predict(fake_img, top_k=5)

print('Top-5 Predictions:')
for i, r in enumerate(results, 1):
    label = r['class'].replace('___', ' — ').replace('_', ' ')
    bar   = '█' * int(r['confidence'] * 40)
    tag   = '✅ Healthy' if r['healthy'] else '🚨 Disease'
    print(f'  {i}. {label:<45} {r["confidence"]*100:5.1f}%  {bar}  {tag}')

## Step 9 — Recommendation Engine

In [ ]:
from modules.recommendation_engine import generate_recommendations, format_report

recs = generate_recommendations(analysis)

print(format_report(field['name'], recs))

## Step 10 — All Fields Summary

In [ ]:
from config import DEMO_FIELDS

summary_rows = []
for f in DEMO_FIELDS:
    print(f"Processing {f['name']}...")
    df_f  = build_ndvi_series(f, days=14)
    wx_f  = fetch_weather(f['lat'], f['lon'])
    ana_f = analyse_field(f, df_f, wx_f)
    summary_rows.append({
        'Field'    : f['name'],
        'Crop'     : f['crop'],
        'NDVI'     : ana_f['ndvi_now'],
        'Z-Score'  : ana_f['zscore']['zscore'],
        'Stress'   : ana_f['zscore']['stress_level'],
        'Risk'     : ana_f['risk']['level'],
        'Temp °C'  : ana_f['weather']['temperature_c'],
        'RH %'     : ana_f['weather']['humidity_pct'],
    })

import pandas as pd
summary_df = pd.DataFrame(summary_rows)
print('\n=== All Fields Summary ===')
print(summary_df.to_string(index=False))

## Step 11 — Launch Streamlit Dashboard

In [ ]:
# In Colab — use pyngrok to tunnel
try:
    from google.colab import drive
    IN_COLAB = True
except:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q pyngrok
    from pyngrok import ngrok
    import subprocess, threading

    proc = subprocess.Popen(['streamlit', 'run', 'app.py',
                              '--server.port', '8501',
                              '--server.headless', 'true'])
    public_url = ngrok.connect(8501)
    print(f'✅ Streamlit is live at: {public_url}')

elif os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
    # Kaggle — use subprocess + display URL
    import subprocess
    subprocess.Popen(['streamlit', 'run', 'app.py', '--server.port', '8501'])
    print('Streamlit started on port 8501')
    print('Use Output tab in Kaggle notebook to see the running service')

else:
    # VS Code / local — just print the command
    print('Run in terminal:')
    print('  streamlit run app.py')
    print('Then open http://localhost:8501')